In [1]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np

# same PROJECT_ROOT you used before
PROJECT_ROOT = Path(r"C:\Users\kjdac\OneDrive\Desktop\MerchantRisk-project")
DB_PATH = PROJECT_ROOT / "db" / "merchant_risk.db"

print("DB exists?", DB_PATH.exists())


DB exists? True


In [2]:
con = sqlite3.connect(DB_PATH)

weekly = pd.read_sql_query("SELECT * FROM weekly_merchant_health;", con)
merchants = pd.read_sql_query("SELECT merchant_id, merchant_name, industry, country, onboarding_channel FROM merchants;", con)

con.close()

weekly["week_start"] = pd.to_datetime(weekly["week_start"])
df = weekly.merge(merchants, on="merchant_id", how="left")

print("Weekly rows:", len(df))
display(df.head())


Weekly rows: 13422


,merchant_id,week_start,txn_total,txn_approved,txn_declined,approval_rate,approved_gmv,avg_approved_amount,refund_count,refund_amount,refund_rate,dispute_count,dispute_amount,dispute_lost_count,dispute_rate,merchant_name,industry,country,onboarding_channel
0,1,2025-08-11,12,10,2,0.8333,2001.20,200.12,0,0.00,0.0000,0,0.0,0,0.0,Merchant_0001,travel,US,sales_assisted
1,1,2025-08-18,19,14,5,0.7368,3928.82,280.63,0,0.00,0.0000,0,0.0,0,0.0,Merchant_0001,travel,US,sales_assisted
2,1,2025-08-25,24,19,5,0.7917,5025.58,264.50,0,0.00,0.0000,0,0.0,0,0.0,Merchant_0001,travel,US,sales_assisted
3,1,2025-09-01,22,19,3,0.8636,3926.66,206.67,3,572.13,0.1579,0,0.0,0,0.0,Merchant_0001,travel,US,sales_assisted
4,1,2025-09-08,24,17,7,0.7083,3405.24,200.31,1,82.95,0.0588,0,0.0,0,0.0,Merchant_0001,travel,US,sales_assisted


In [3]:
df = df.sort_values(["merchant_id", "week_start"]).copy()

metrics = ["approval_rate", "refund_rate", "dispute_rate", "approved_gmv"]

for m in metrics:
    # rolling mean of the previous 4 weeks (shift(1) prevents using the current week)
    df[f"{m}_baseline_4w"] = (
        df.groupby("merchant_id")[m]
          .shift(1)
          .rolling(window=4, min_periods=2)
          .mean()
    )

print("✅ Baselines created.")
display(df[[ "merchant_id","week_start","approval_rate","approval_rate_baseline_4w"]].head(10))


✅ Baselines created.


,merchant_id,week_start,approval_rate,approval_rate_baseline_4w
0,1,2025-08-11,0.8333,NaN
1,1,2025-08-18,0.7368,NaN
2,1,2025-08-25,0.7917,0.785050
3,1,2025-09-01,0.8636,0.787267
4,1,2025-09-08,0.7083,0.806350
5,1,2025-09-15,0.8750,0.775100
6,1,2025-09-22,0.8000,0.809650
7,1,2025-09-29,0.8750,0.811725
8,1,2025-10-06,0.6923,0.814575
9,1,2025-10-13,0.7273,0.810575


In [4]:
def safe_divide(a, b):
    return np.where((b == 0) | pd.isna(b), np.nan, a / b)

# deltas vs baseline
df["refund_mult"] = safe_divide(df["refund_rate"], df["refund_rate_baseline_4w"])
df["dispute_mult"] = safe_divide(df["dispute_rate"], df["dispute_rate_baseline_4w"])
df["gmv_mult"] = safe_divide(df["approved_gmv"], df["approved_gmv_baseline_4w"])

df["approval_drop"] = df["approval_rate_baseline_4w"] - df["approval_rate"]  # positive = dropped

# basic thresholds (tuneable)
df["alert_refund_spike"] = (df["refund_rate_baseline_4w"].notna()) & (df["refund_mult"] >= 2.0) & (df["refund_rate"] >= 0.02)
df["alert_dispute_spike"] = (df["dispute_rate_baseline_4w"].notna()) & (df["dispute_mult"] >= 2.0) & (df["dispute_rate"] >= 0.01)
df["alert_approval_drop"] = (df["approval_rate_baseline_4w"].notna()) & (df["approval_drop"] >= 0.10)
df["alert_gmv_surge"] = (df["approved_gmv_baseline_4w"].notna()) & (df["gmv_mult"] >= 2.0) & (df["approved_gmv"] >= 5000)

print("✅ Alerts flags created.")
print(df[["alert_refund_spike","alert_dispute_spike","alert_approval_drop","alert_gmv_surge"]].sum())


✅ Alerts flags created.
alert_refund_spike     1401
alert_dispute_spike     526
alert_approval_drop    2530
alert_gmv_surge          21
dtype: int64


In [5]:
alert_rows = []

def severity_from_flags(row):
    # simple severity logic:
    # High if dispute spike OR (refund spike + approval drop), else Medium
    if row["alert_dispute_spike"]:
        return "High"
    if row["alert_refund_spike"] and row["alert_approval_drop"]:
        return "High"
    return "Medium"

def reason_list(row):
    reasons = []
    if row["alert_dispute_spike"]:
        reasons.append("dispute_rate_spike")
    if row["alert_refund_spike"]:
        reasons.append("refund_rate_spike")
    if row["alert_approval_drop"]:
        reasons.append("approval_rate_drop")
    if row["alert_gmv_surge"]:
        reasons.append("gmv_surge")
    return ", ".join(reasons)

for _, r in df.iterrows():
    any_alert = r["alert_refund_spike"] or r["alert_dispute_spike"] or r["alert_approval_drop"] or r["alert_gmv_surge"]
    if not any_alert:
        continue

    alert_rows.append({
        "week_start": r["week_start"].date().isoformat(),
        "merchant_id": int(r["merchant_id"]),
        "merchant_name": r["merchant_name"],
        "industry": r["industry"],
        "country": r["country"],
        "onboarding_channel": r["onboarding_channel"],

        "severity": severity_from_flags(r),
        "reasons": reason_list(r),

        # current metrics
        "approval_rate": float(r["approval_rate"]),
        "refund_rate": float(r["refund_rate"]),
        "dispute_rate": float(r["dispute_rate"]),
        "approved_gmv": float(r["approved_gmv"]),

        # baselines
        "approval_rate_baseline_4w": float(r["approval_rate_baseline_4w"]) if pd.notna(r["approval_rate_baseline_4w"]) else np.nan,
        "refund_rate_baseline_4w": float(r["refund_rate_baseline_4w"]) if pd.notna(r["refund_rate_baseline_4w"]) else np.nan,
        "dispute_rate_baseline_4w": float(r["dispute_rate_baseline_4w"]) if pd.notna(r["dispute_rate_baseline_4w"]) else np.nan,
        "approved_gmv_baseline_4w": float(r["approved_gmv_baseline_4w"]) if pd.notna(r["approved_gmv_baseline_4w"]) else np.nan,

        # change signals
        "refund_mult": float(r["refund_mult"]) if pd.notna(r["refund_mult"]) else np.nan,
        "dispute_mult": float(r["dispute_mult"]) if pd.notna(r["dispute_mult"]) else np.nan,
        "gmv_mult": float(r["gmv_mult"]) if pd.notna(r["gmv_mult"]) else np.nan,
        "approval_drop": float(r["approval_drop"]) if pd.notna(r["approval_drop"]) else np.nan,
    })

alerts = pd.DataFrame(alert_rows)

print("Alerts found:", len(alerts))
display(alerts.head())


Alerts found: 4064


,week_start,merchant_id,merchant_name,industry,country,onboarding_channel,severity,reasons,approval_rate,refund_rate,dispute_rate,approved_gmv,approval_rate_baseline_4w,refund_rate_baseline_4w,dispute_rate_baseline_4w,approved_gmv_baseline_4w,refund_mult,dispute_mult,gmv_mult,approval_drop
0,2025-09-15,1,Merchant_0001,travel,US,sales_assisted,Medium,refund_rate_spike,0.8750,0.2143,0.0000,2251.48,0.775100,0.054175,0.000000,4071.575,3.955699,NaN,0.552975,-0.099900
1,2025-10-06,1,Merchant_0001,travel,US,sales_assisted,Medium,approval_rate_drop,0.6923,0.0000,0.0000,2182.98,0.814575,0.083900,0.015625,3622.805,0.000000,0.000000,0.602566,0.122275
2,2025-10-20,1,Merchant_0001,travel,US,sales_assisted,High,dispute_rate_spike,0.8000,0.0000,0.0833,3485.75,0.773650,0.015625,0.015625,3295.965,0.000000,5.331200,1.057581,-0.026350
3,2025-10-27,1,Merchant_0001,travel,US,sales_assisted,High,dispute_rate_spike,0.8400,0.0476,0.0476,4296.98,0.773650,0.000000,0.020825,3186.795,NaN,2.285714,1.348370,-0.066350
4,2025-11-10,1,Merchant_0001,travel,US,sales_assisted,High,"refund_rate_spike, approval_rate_drop",0.7000,0.0714,0.0000,2325.24,0.815500,0.011900,0.032725,3137.360,6.000000,0.000000,0.741145,0.115500


In [6]:
# Sort: High first, then most extreme disputes/refunds
severity_order = {"High": 0, "Medium": 1}
alerts["severity_rank"] = alerts["severity"].map(severity_order)

alerts = alerts.sort_values(
    ["severity_rank", "week_start", "dispute_rate", "refund_rate"],
    ascending=[True, False, False, False]
).drop(columns=["severity_rank"])

out_path = PROJECT_ROOT / "outputs" / "alerts_queue.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)

alerts.to_csv(out_path, index=False)
print("✅ Saved:", out_path)


✅ Saved: C:\Users\kjdac\OneDrive\Desktop\MerchantRisk-project\outputs\alerts_queue.csv


In [7]:
print(alerts["severity"].value_counts())
print("Most common reasons:")
print(alerts["reasons"].value_counts().head(10))


severity
Medium    3320
High       744
Name: count, dtype: int64
Most common reasons:
reasons
approval_rate_drop                                           2204
refund_rate_spike                                            1096
dispute_rate_spike                                            349
refund_rate_spike, approval_rate_drop                         218
dispute_rate_spike, approval_rate_drop                         91
dispute_rate_spike, refund_rate_spike                          70
gmv_surge                                                      16
dispute_rate_spike, refund_rate_spike, approval_rate_drop      15
refund_rate_spike, gmv_surge                                    2
approval_rate_drop, gmv_surge                                   2
Name: count, dtype: int64
